# Clean Data Science Salaries Dataset

This notebook imports the raw salary dataset, applies validation and cleaning rules, and saves the processed data.

In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
RAW_PATH = PROJECT_ROOT / 'data' / 'raw' / 'ds_salaries.csv'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
CLEAN_PATH = PROCESSED_DIR / 'ds_salaries_clean.csv'
REPORT_PATH = PROCESSED_DIR / 'cleaning_report.csv'

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(RAW_PATH)
print(f'Imported {df.shape[0]:,} rows and {df.shape[1]} columns from {RAW_PATH.name}')

Imported 607 rows and 12 columns from ds_salaries.csv


In [ ]:
# Remove the CSV export index and normalize column names.
df = df.loc[:, ~df.columns.str.match(r'^Unnamed')]
df.columns = df.columns.str.strip().str.lower()

text_columns = df.select_dtypes(include='object').columns
df[text_columns] = df[text_columns].apply(lambda column: column.str.strip())

numeric_columns = ['work_year', 'salary', 'salary_in_usd', 'remote_ratio']
df[numeric_columns] = df[numeric_columns].apply(pd.to_numeric, errors='coerce')

required_columns = [
    'work_year', 'experience_level', 'employment_type', 'job_title',
    'salary', 'salary_currency', 'salary_in_usd', 'employee_residence',
    'remote_ratio', 'company_location', 'company_size'
]
missing_required = df[required_columns].isna().any(axis=1)
invalid_salary = (df['salary'] <= 0) | (df['salary_in_usd'] <= 0)
invalid_remote_ratio = ~df['remote_ratio'].isin([0, 50, 100])

rows_before = len(df)
duplicate_rows = int(df.duplicated().sum())
invalid_rows = missing_required | invalid_salary | invalid_remote_ratio
df = df.loc[~invalid_rows].drop_duplicates().reset_index(drop=True)

cleaning_report = pd.DataFrame({
    'metric': ['rows imported', 'duplicate rows removed', 'invalid rows removed', 'rows exported'],
    'value': [rows_before, duplicate_rows, int(invalid_rows.sum()), len(df)]
})

df.to_csv(CLEAN_PATH, index=False)
cleaning_report.to_csv(REPORT_PATH, index=False)
print(cleaning_report.to_string(index=False))
print(f'\nSaved cleaned data to {CLEAN_PATH}')

In [2]:
assert not df.isna().any().any(), 'Cleaned data contains missing values'
assert df['salary'].gt(0).all() and df['salary_in_usd'].gt(0).all()
assert df['remote_ratio'].isin([0, 50, 100]).all()
assert not df.duplicated().any()

display(df.head())
print(f'Final shape: {df.shape}')
print(f'Missing values: {int(df.isna().sum().sum())}')

,Unnamed: 0,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,remote_ratio,company_location,company_size
0,0,2020,MI,FT,Data Scientist,70000,EUR,79833,DE,0,DE,L
1,1,2020,SE,FT,Machine Learning Scientist,260000,USD,260000,JP,0,JP,S
2,2,2020,SE,FT,Big Data Engineer,85000,GBP,109024,GB,50,GB,M
3,3,2020,MI,FT,Product Data Analyst,20000,USD,20000,HN,0,HN,S
4,4,2020,SE,FT,Machine Learning Engineer,150000,USD,150000,US,50,US,L


Final shape: (607, 12)
Missing values: 0
